# Detective R-GNN — Stage 2: PPO fine-tuning

Carica il best BC checkpoint come warm start e lo affina con PPO contro MCTS Mr.X a piena potenza.

**Setup** (vedi `DESIGN.md` §9):
- Rollout: sampling dalla categorical sulla policy edge-conditioned
- Eval: argmax (paired vs heuristic, stessa procedura di `colab_warmstart_eval.ipynb`)
- 5 detective decisions per game-turn = up to 5 transitions. La team-reward `r_t` viene divisa fra le transizioni effettive del turno, cosi' la somma per game-turn resta coerente con il `return_to_go` del BC.
- γ = 0.95 (per step), GAE λ = 0.95, clip ε = 0.2, entropy 0.01, value coef 0.5, LR 3e-4.
- Mr. X = full strength (`explo=15, sims=25`).

**Cosa monitorare**:
1. Detective winrate vs heuristic durante il training (cresce o crolla?)
2. Policy entropy (deve scendere lentamente, non collassare)
3. Value loss + policy loss
4. KL approx tra old e new policy (deve stare sotto 0.02-0.03)

## 1. Setup

In [ ]:
%%bash
set -e
cd /content
if [ ! -d scotland_yard ]; then
  git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git
fi
ls scotland_yard

In [ ]:
!pip install -q networkx numpy pandas tqdm scipy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DRIVE_BASE = '/content/drive/MyDrive/scotland_yard_dataset'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')
PPO_CKPT_DIR = os.path.join(DRIVE_BASE, 'ppo_checkpoints')
os.makedirs(PPO_CKPT_DIR, exist_ok=True)

bc_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'rgnn_bc_best_*.pt')))
assert bc_ckpts, f'Nessun BC checkpoint in {CHECKPOINT_DIR}'
BC_CKPT_PATH = bc_ckpts[-1]
print('BC warm-start checkpoint:', BC_CKPT_PATH)
print('PPO checkpoints output:', PPO_CKPT_DIR)

## 2. Imports + repo

In [ ]:
import os, sys, json, time, random, math
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

REPO_DIR = '/content/scotland_yard'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

from game import Game, ADJ, BOARD_GRAPH, SHORTEST_PATH_TENSOR, ticket_bitmask
from detective_engine import DetectiveEngine
import mrx_engine
from mrx_engine import MrxEngine
from utility import _min_detective_distance

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 3. Static graph + reveal schedule + reward constants

In [ ]:
N_NODES = 199
RELATIONS = ('taxi', 'bus', 'underground', 'water')
REL2IDX = {r: i for i, r in enumerate(RELATIONS)}
VEHICLES = ('taxi', 'bus', 'underground')
VEH2IDX = {v: i for i, v in enumerate(VEHICLES)}
TICKET_VOCAB = {'': 0, 'taxi': 1, 'bus': 2, 'underground': 3, 'water': 4}
MAX_TURN = 22
REVEAL_TURNS = (3, 8, 13, 18)

# Reward (design doc §5)
GAMMA = 0.95
GAE_LAMBDA = 0.95
R_CAPTURE = +10.0
R_TIMEOUT = -10.0
R_STEP = -0.05
R_DIST_COEF = -0.1


def turns_to_next_reveal(t):
    for r in REVEAL_TURNS:
        if r > t: return r - t
    return -1


def build_dense_adj():
    A = np.zeros((len(RELATIONS), N_NODES, N_NODES), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            for t in types:
                if t in REL2IDX:
                    A[REL2IDX[t], v, u] = 1.0
    deg = A.sum(axis=2, keepdims=True); deg[deg == 0] = 1.0
    return A / deg

def build_node_static_features():
    deg = np.zeros((N_NODES, len(RELATIONS)), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            for t in types:
                if t in REL2IDX:
                    deg[v, REL2IDX[t]] += 1.0
    has_und = (deg[:, REL2IDX['underground']] > 0).astype(np.float32)[:, None]
    return np.concatenate([deg, has_und], axis=1)

def build_edge_lookup():
    lookup = {}
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        out = []
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            rel_idx = sorted({REL2IDX[t] for t in types if t in REL2IDX})
            out.append((u, rel_idx))
        lookup[v] = out
    return lookup

DENSE_ADJ = torch.from_numpy(build_dense_adj()).to(DEVICE)
NODE_STATIC = torch.from_numpy(build_node_static_features()).to(DEVICE)
EDGE_LOOKUP = build_edge_lookup()
_SP_ANY_NP = SHORTEST_PATH_TENSOR[7, :199, :199].astype(np.float32)

NODE_DYN_DIM = 1 + 5 + 1 + 1 + 5 + 1   # 14
GLOBAL_FEATURE_DIM = 1 + 1 + 4 + 15 + 5  # 26
print('Graph OK. DENSE_ADJ', DENSE_ADJ.shape, 'NODE_STATIC', NODE_STATIC.shape)

## 4. Model architecture (identica al training)

In [ ]:
class DenseRGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_relations, dropout=0.1):
        super().__init__()
        self.W0 = nn.Linear(in_dim, out_dim, bias=True)
        self.Wr = nn.Parameter(torch.empty(n_relations, in_dim, out_dim))
        nn.init.xavier_uniform_(self.Wr)
        self.dropout = nn.Dropout(dropout)
        self.residual = (in_dim == out_dim)
    def forward(self, h, A):
        self_msg = self.W0(h)
        AH = torch.einsum('rmn,bnk->rbmk', A, h)
        AHW = torch.einsum('rbmk,rkj->rbmj', AH, self.Wr)
        rel_msg = AHW.sum(dim=0)
        out = F.relu(self_msg + rel_msg)
        out = self.dropout(out)
        if self.residual: out = out + h
        return out


class DetectiveRGNN(nn.Module):
    def __init__(self, node_dyn_dim, node_static_dim, global_dim,
                 hidden=64, n_layers=3, n_relations=4, det_emb_dim=8,
                 edge_emb_dim=8, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(node_dyn_dim + node_static_dim, hidden)
        self.layers = nn.ModuleList([
            DenseRGCNLayer(hidden, hidden, n_relations, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.det_id_emb = nn.Embedding(5, det_emb_dim)
        self.edge_type_emb = nn.Embedding(n_relations, edge_emb_dim)

        ph_in = 2*hidden + edge_emb_dim + global_dim + det_emb_dim
        self.policy_mlp = nn.Sequential(
            nn.Linear(ph_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        vh_in = 2*hidden + global_dim + det_emb_dim
        self.value_mlp = nn.Sequential(
            nn.Linear(vh_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, node_dyn, node_static_b, A):
        x = torch.cat([node_dyn, node_static_b], dim=-1)
        h = self.input_proj(x)
        for layer in self.layers:
            h = layer(h, A)
        return h

    def forward(self, batch, A, node_static):
        """Returns: value (B,), all_logits list len B, all_cand list len B."""
        B = batch['node_dyn'].shape[0]
        node_dyn = batch['node_dyn']
        glob = batch['glob']
        det_id = batch['det_id']
        ego_pos = batch['ego_pos']
        legal_neigh = batch['legal_neighbors_mask']

        node_static_b = node_static.unsqueeze(0).expand(B, -1, -1)
        h = self.encode(node_dyn, node_static_b, A)
        det_e = self.det_id_emb(det_id)
        glob_full = torch.cat([glob, det_e], dim=-1)

        h_mean = h.mean(dim=1)
        h_max = h.max(dim=1).values
        v_in = torch.cat([h_mean, h_max, glob_full], dim=-1)
        value = self.value_mlp(v_in).squeeze(-1)

        all_logits, all_cand = [], []
        for b in range(B):
            v = int(ego_pos[b].item())
            mask = legal_neigh[b].cpu().numpy() if legal_neigh.is_cuda else legal_neigh[b].numpy()
            cand = [(u, rels) for (u, rels) in EDGE_LOOKUP[v]
                    if u < N_NODES and mask[u]]
            if not cand:
                all_logits.append(torch.zeros(1, device=h.device))
                all_cand.append([])
                continue
            us = torch.tensor([u for (u, _) in cand], device=h.device, dtype=torch.long)
            edge_e = torch.stack([self.edge_type_emb(
                torch.tensor(rels, device=h.device, dtype=torch.long)).sum(dim=0)
                for (_, rels) in cand], dim=0)
            h_v = h[b, v].unsqueeze(0).expand(len(cand), -1)
            h_u = h[b, us]
            g_b = glob_full[b].unsqueeze(0).expand(len(cand), -1)
            feat = torch.cat([h_v, h_u, edge_e, g_b], dim=-1)
            logits = self.policy_mlp(feat).squeeze(-1)
            all_logits.append(logits)
            all_cand.append([u for (u, _) in cand])
        return value, all_logits, all_cand

## 5. Load BC checkpoint come warm start

In [ ]:
ckpt = torch.load(BC_CKPT_PATH, map_location=DEVICE, weights_only=False)
cfg = ckpt.get('config', {})
print('BC checkpoint metadata:')
for k in ('epoch', 'val_nll', 'val_acc', 'rtg_mean', 'rtg_std'):
    if k in ckpt: print(f'  {k}: {ckpt[k]}')

model = DetectiveRGNN(
    node_dyn_dim=cfg.get('node_dyn_dim', NODE_DYN_DIM),
    node_static_dim=cfg.get('node_static_dim', int(NODE_STATIC.shape[1])),
    global_dim=cfg.get('global_dim', GLOBAL_FEATURE_DIM),
    hidden=cfg.get('hidden', 64),
    n_layers=cfg.get('n_layers', 3),
    n_relations=cfg.get('n_relations', 4),
    dropout=0.1,
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel loaded with BC weights: {n_params/1e3:.1f}k params')

# rtg normalization stats — il value head BC predice rtg NORMALIZZATI.
# In PPO produrremo target di GAE su scala raw; serve denormalizzazione coerente.
BC_RTG_MEAN = float(ckpt.get('rtg_mean', 0.0))
BC_RTG_STD = float(ckpt.get('rtg_std', 1.0))
print(f'BC value head was trained on normalized rtg: mean={BC_RTG_MEAN:.3f} std={BC_RTG_STD:.3f}')

## 6. Build-input helper + reward function

`build_input_dict(game, engine, det_id, revealed_positions, last_mrx_ticket)` produce le stesse feature usate in training/eval. Reward function identica al logger.

In [ ]:
def build_input_dict(game, engine, det_id, revealed_positions=(), last_mrx_ticket=''):
    """Costruisce gli input tensors per una singola decisione detective.
    Tutte le feature osservabili: niente leakage di mrx_pos o mrx_moves."""
    mrx_visited_mask = np.zeros(N_NODES, dtype=np.float32)
    for p in revealed_positions:
        mrx_visited_mask[int(p) - 1] = 1.0

    det_pos_arr = np.array([int(p) - 1 for p in game.detectives_pos], dtype=np.int64)
    ego = det_pos_arr[det_id]
    belief = engine.belief_state.astype(np.float32)
    is_det = np.zeros((5, N_NODES), dtype=np.float32)
    for k, p in enumerate(det_pos_arr): is_det[k, p] = 1.0
    is_ego = np.zeros(N_NODES, dtype=np.float32); is_ego[ego] = 1.0
    dpd = np.stack([_SP_ANY_NP[p] for p in det_pos_arr], axis=1).astype(np.float32)
    md = dpd.min(axis=1, keepdims=True)
    node_dyn = np.concatenate([
        belief[:, None], is_det.T, is_ego[:, None], md, dpd, mrx_visited_mask[:, None]
    ], axis=1).astype(np.float32)

    legal = np.zeros((N_NODES, 3), dtype=bool)
    occupied = set(game.detectives_pos[:det_id] + game.detectives_pos[det_id + 1:])
    tickets = game.detective_tickets[det_id]
    for nb_, types in ADJ[game.detectives_pos[det_id]].items():
        if nb_ in occupied: continue
        for t in types:
            if t in VEH2IDX and tickets.get(t, 0) > 0:
                legal[int(nb_) - 1, VEH2IDX[t]] = True
    legal_neigh = legal.any(axis=1)

    last_oh = np.zeros(5, dtype=np.float32); last_oh[TICKET_VOCAB.get(last_mrx_ticket, 0)] = 1.0
    all_dt = np.array([[t['taxi'], t['bus'], t['underground']] for t in game.detective_tickets],
                      dtype=np.float32) / 10.0
    glob = np.concatenate([
        np.array([game.turn / MAX_TURN, (turns_to_next_reveal(game.turn) / 5.0)], dtype=np.float32),
        np.array([game.mrx_tickets[v] for v in ('taxi','bus','underground','water')],
                 dtype=np.float32) / 10.0,
        all_dt.reshape(-1),
        last_oh,
    ])
    return {
        'node_dyn': node_dyn.astype(np.float32),
        'glob': glob.astype(np.float32),
        'det_id': int(det_id),
        'ego_pos': int(ego),
        'legal_neighbors_mask': legal_neigh,
    }


def per_turn_reward(detectives_pos, mrx_pos, captured, timeout):
    """Reward del game-turn (design doc §5, cooperative). Calcolata DOPO la mossa di Mr.X.
    detectives_pos e mrx_pos sono 1-indexed."""
    r = R_STEP
    if captured:
        r += R_CAPTURE
        return r  # min_dist = 0 by def
    md = _min_detective_distance(detectives_pos, mrx_pos)
    r += R_DIST_COEF * md
    if timeout:
        r += R_TIMEOUT
    return r

## 7. Rollout collection (sampling)

Una partita = lista di transizioni. Per ogni decisione detective salva: input_dict, action_idx, log_prob, value, e (dopo il turno) reward + done. La reward del game-turn `r_t` viene divisa tra le transizioni effettivamente raccolte in quel turno: la somma per turno resta uguale al reward del logger BC, ma il credito resta distribuito tra i detective.

In [ ]:
def collate_one(s):
    """Crea un batch di dimensione 1 dai numpy arrays di build_input_dict."""
    return {
        'node_dyn': torch.from_numpy(s['node_dyn']).unsqueeze(0),
        'glob': torch.from_numpy(s['glob']).unsqueeze(0),
        'det_id': torch.tensor([s['det_id']], dtype=torch.long),
        'ego_pos': torch.tensor([s['ego_pos']], dtype=torch.long),
        'legal_neighbors_mask': torch.from_numpy(s['legal_neighbors_mask']).unsqueeze(0),
    }


def assign_turn_reward(transitions, turn_start_idx, reward):
    """Divide il reward cooperativo del game-turn tra le decisioni registrate nel turno.
    Mantiene la scala dei return vicina al BC logger, dove c'e' un solo reward per game-turn."""
    turn_ts = transitions[turn_start_idx:]
    if not turn_ts:
        return
    per_decision_reward = reward / len(turn_ts)
    for t in turn_ts:
        t['reward'] = per_decision_reward


@torch.no_grad()
def collect_episode(model, mrx_explo=15, mrx_sims=25):
    """Una partita. Detectives campionano dalla policy. Mr.X = MCTS.
    Ritorna lista di transizioni dict con keys:
      input (dict numpy), action_idx, log_prob, value, reward, done.
    """
    mrx_engine.NUM_EXPLORATIONS = mrx_explo
    mrx_engine.NUM_SIMULATIONS = mrx_sims

    model.eval()
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)

    revealed_positions = []
    last_mrx_ticket = ''
    transitions = []   # buffer per l'intero episodio
    turn_start_idx = 0  # primo idx di transizione del game-turn corrente

    while True:
        if game.check_victory(silent=True):
            break

        game_over = False
        for det_id in range(game.num_detectives):
            s = build_input_dict(game, engine, det_id, tuple(revealed_positions), last_mrx_ticket)
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in collate_one(s).items()}
            value, all_logits, all_cand = model(batch, DENSE_ADJ, NODE_STATIC)
            logits = all_logits[0]
            cand = all_cand[0]

            if not cand:
                # detective bloccato: niente transizione registrata
                if game.check_victory(silent=True): game_over = True; break
                continue

            log_probs = F.log_softmax(logits, dim=-1)
            probs = log_probs.exp()
            action_idx = int(torch.multinomial(probs, 1).item())
            lp = float(log_probs[action_idx].item())

            best_dest = str(int(cand[action_idx]) + 1)   # Game/ADJ usano nodi stringa
            pos = game.detectives_pos[det_id]
            game.use_ticket(det_id, pos, best_dest)
            game.detectives_pos[det_id] = best_dest

            transitions.append({
                'input': s,
                'action_idx': action_idx,
                'log_prob': lp,
                'value': float(value[0].item()) * BC_RTG_STD + BC_RTG_MEAN,  # denormalizza V(s)
                'reward': 0.0,
                'done': False,
            })

            if game.check_victory(silent=True): game_over = True; break
            engine.kalman_filter()

        game.detectives_moves.append(game.detectives_pos[:])
        if game_over:
            # cattura durante il detective phase: reward = +10 + step (md=0)
            r = per_turn_reward(game.detectives_pos, game.mrx_pos, captured=True, timeout=False)
            assign_turn_reward(transitions, turn_start_idx, r)
            if transitions: transitions[-1]['done'] = True
            break

        # Mr.X turn
        best_pos, best_ticket = mrx.search()
        game.x_automated_turn(best_pos, best_ticket)
        if best_ticket is not None:
            last_mrx_ticket = best_ticket

        if game.check_victory(silent=True):
            captured = (game.winner == 0)
            timeout = (game.winner == 1)
            r = per_turn_reward(game.detectives_pos, game.mrx_pos, captured=captured, timeout=timeout)
            assign_turn_reward(transitions, turn_start_idx, r)
            if transitions: transitions[-1]['done'] = True
            break

        engine.update_belief_after_mrx_move(best_ticket)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
            revealed_positions.append(int(game.mrx_pos))

        # Game-turn completo: divide il reward sulle transizioni del turno
        r = per_turn_reward(game.detectives_pos, game.mrx_pos, captured=False, timeout=False)
        assign_turn_reward(transitions, turn_start_idx, r)
        turn_start_idx = len(transitions)

    return transitions, int(game.winner)

## 8. GAE + RolloutBuffer

Calcola advantage e return per ogni transizione. Bootstrap finale 0 se `done`, altrimenti V(s_T) (qui usiamo 0 perché tutti gli episodi terminano).

In [ ]:
def compute_gae(transitions, gamma=GAMMA, lam=GAE_LAMBDA, last_value=0.0):
    """Aggiunge 'advantage' e 'return' a ogni transition. In-place."""
    n = len(transitions)
    if n == 0: return
    advantages = [0.0] * n
    gae = 0.0
    next_v = last_value
    next_done = 1.0  # episodio terminato
    for t in range(n - 1, -1, -1):
        r = transitions[t]['reward']
        v = transitions[t]['value']
        done = transitions[t]['done']
        not_terminal = 1.0 - (1.0 if done else 0.0)
        delta = r + gamma * next_v * not_terminal - v
        gae = delta + gamma * lam * not_terminal * gae
        advantages[t] = gae
        next_v = v
    for t in range(n):
        transitions[t]['advantage'] = advantages[t]
        transitions[t]['return'] = advantages[t] + transitions[t]['value']


class RolloutBuffer:
    """Accumula transizioni e produce batch tensors per il PPO update."""
    def __init__(self):
        self.transitions = []
    def add_episode(self, transitions, last_value=0.0):
        compute_gae(transitions, last_value=last_value)
        self.transitions.extend(transitions)
    def __len__(self): return len(self.transitions)
    def clear(self): self.transitions = []

    def to_tensors(self, device):
        """Stack di tensori per il batch update PPO."""
        ts = self.transitions
        node_dyn = torch.from_numpy(np.stack([t['input']['node_dyn'] for t in ts])).to(device)
        glob = torch.from_numpy(np.stack([t['input']['glob'] for t in ts])).to(device)
        det_id = torch.tensor([t['input']['det_id'] for t in ts], dtype=torch.long, device=device)
        ego_pos = torch.tensor([t['input']['ego_pos'] for t in ts], dtype=torch.long, device=device)
        legal_neigh = torch.from_numpy(np.stack([t['input']['legal_neighbors_mask'] for t in ts])).to(device)
        actions = torch.tensor([t['action_idx'] for t in ts], dtype=torch.long, device=device)
        old_log_probs = torch.tensor([t['log_prob'] for t in ts], dtype=torch.float, device=device)
        advantages = torch.tensor([t['advantage'] for t in ts], dtype=torch.float, device=device)
        returns = torch.tensor([t['return'] for t in ts], dtype=torch.float, device=device)
        return {
            'node_dyn': node_dyn, 'glob': glob, 'det_id': det_id, 'ego_pos': ego_pos,
            'legal_neighbors_mask': legal_neigh, 'actions': actions,
            'old_log_probs': old_log_probs, 'advantages': advantages, 'returns': returns,
        }

### Nota tecnica: scala del value head

Il logger BC calcola un solo reward cooperativo per game-turn e assegna lo stesso `return_to_go` alle decisioni dei detective prese dentro quel turno.

Conseguenze pratiche:
In PPO dividiamo quel reward tra le transizioni effettivamente raccolte nel turno. Cosi' la somma dei reward per game-turn resta sulla scala del BC, mentre il credito e' ancora distribuito sulle azioni sequenziali dei detective.

Se vuoi sperimentare una reward shared non scalata, puoi rimuovere `assign_turn_reward`, ma allora conviene abbassare `PPO_VALUE_COEF` per non far dominare la loss del value head.

## 9. PPO update step

Standard PPO: clipped surrogate + value MSE + entropy bonus. Su variable-action-space il log_prob è ricomputato dal cand list (deterministico).

In [ ]:
PPO_CLIP_EPS = 0.2
PPO_ENTROPY_COEF = 0.01
PPO_VALUE_COEF = 0.5
PPO_GRAD_CLIP = 1.0


def ppo_update(model, optimizer, buf_tensors, n_epochs=4, minibatch_size=128):
    """Esegue n_epochs di update PPO sul buffer. Ritorna metriche medie."""
    n = buf_tensors['actions'].shape[0]
    if n == 0:
        return {'pi_loss': 0, 'v_loss': 0, 'entropy': 0, 'approx_kl': 0, 'clip_frac': 0}

    # Normalizza advantages per ridurre la varianza
    adv = buf_tensors['advantages']
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    # Per il value head normalizziamo i target nella stessa scala usata dal BC
    returns_norm = (buf_tensors['returns'] - BC_RTG_MEAN) / BC_RTG_STD

    metrics = defaultdict(list)
    model.train()
    for _ in range(n_epochs):
        perm = torch.randperm(n, device=DEVICE)
        for start in range(0, n, minibatch_size):
            idx = perm[start:start + minibatch_size]
            mb = {
                'node_dyn': buf_tensors['node_dyn'][idx],
                'glob': buf_tensors['glob'][idx],
                'det_id': buf_tensors['det_id'][idx],
                'ego_pos': buf_tensors['ego_pos'][idx],
                'legal_neighbors_mask': buf_tensors['legal_neighbors_mask'][idx],
            }
            mb_actions = buf_tensors['actions'][idx]
            mb_old_lp = buf_tensors['old_log_probs'][idx]
            mb_adv = adv[idx]
            mb_ret_norm = returns_norm[idx]

            value, all_logits, all_cand = model(mb, DENSE_ADJ, NODE_STATIC)
            # Computa new_log_prob ed entropy per sample
            new_lp_list, ent_list = [], []
            for b in range(len(all_logits)):
                lg = all_logits[b]
                a = int(mb_actions[b].item())
                if len(lg) == 0 or a >= len(lg):
                    # transition con cand vuoto: skip
                    new_lp_list.append(torch.zeros((), device=DEVICE))
                    ent_list.append(torch.zeros((), device=DEVICE))
                    continue
                log_p = F.log_softmax(lg, dim=-1)
                p = log_p.exp()
                new_lp_list.append(log_p[a])
                ent_list.append(-(p * log_p).sum())
            new_lp = torch.stack(new_lp_list)
            entropy = torch.stack(ent_list).mean()

            logratio = new_lp - mb_old_lp
            ratio = logratio.exp()
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1.0 - PPO_CLIP_EPS, 1.0 + PPO_CLIP_EPS) * mb_adv
            pi_loss = -torch.min(surr1, surr2).mean()
            v_loss = F.mse_loss(value, mb_ret_norm)
            loss = pi_loss + PPO_VALUE_COEF * v_loss - PPO_ENTROPY_COEF * entropy

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), PPO_GRAD_CLIP)
            optimizer.step()

            with torch.no_grad():
                approx_kl = ((ratio - 1.0) - logratio).mean().item()
                clip_frac = ((ratio - 1.0).abs() > PPO_CLIP_EPS).float().mean().item()
            metrics['pi_loss'].append(pi_loss.item())
            metrics['v_loss'].append(v_loss.item())
            metrics['entropy'].append(entropy.item())
            metrics['approx_kl'].append(approx_kl)
            metrics['clip_frac'].append(clip_frac)
    return {k: float(np.mean(v)) for k, v in metrics.items()}

## 10. Periodic eval (argmax, paired vs heuristic)

Stesso protocollo di `colab_warmstart_eval.ipynb`: paired seed, argmax. Misura il winrate detective e confronta con la heuristic ad ogni `EVAL_EVERY` update.

In [ ]:
@torch.no_grad()
def policy_act_argmax(model, game, engine, det_id, revealed_positions=(), last_mrx_ticket=''):
    s = build_input_dict(game, engine, det_id, revealed_positions, last_mrx_ticket)
    batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in collate_one(s).items()}
    _, all_logits, all_cand = model(batch, DENSE_ADJ, NODE_STATIC)
    cand = all_cand[0]
    if not cand: return None
    best_idx = int(torch.argmax(all_logits[0]).item())
    best_dest = str(int(cand[best_idx]) + 1)
    return best_dest


def play_one_argmax(model, mrx_explo=15, mrx_sims=25):
    """Una partita con detective in argmax. Restituisce winner."""
    mrx_engine.NUM_EXPLORATIONS = mrx_explo
    mrx_engine.NUM_SIMULATIONS = mrx_sims
    model.eval()
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)
    revealed_positions, last_mrx_ticket = [], ''
    while True:
        if game.check_victory(silent=True): break
        game_over = False
        for det_id in range(game.num_detectives):
            best_dest = policy_act_argmax(model, game, engine, det_id,
                                              tuple(revealed_positions), last_mrx_ticket)
            if best_dest is None:
                if game.check_victory(silent=True): game_over = True; break
                continue
            pos = game.detectives_pos[det_id]
            game.use_ticket(det_id, pos, best_dest)
            game.detectives_pos[det_id] = best_dest
            if game.check_victory(silent=True): game_over = True; break
            engine.kalman_filter()
        game.detectives_moves.append(game.detectives_pos[:])
        if game_over: break
        bp, bt = mrx.search()
        game.x_automated_turn(bp, bt)
        if bt is not None: last_mrx_ticket = bt
        if game.check_victory(silent=True): break
        engine.update_belief_after_mrx_move(bt)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
            revealed_positions.append(int(game.mrx_pos))
    return game.winner


def play_heuristic(mrx_explo=15, mrx_sims=25):
    mrx_engine.NUM_EXPLORATIONS = mrx_explo
    mrx_engine.NUM_SIMULATIONS = mrx_sims
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)
    while True:
        if game.check_victory(silent=True): break
        game_over = False
        for i in range(game.num_detectives):
            game.detective_automated_turn(engine.belief_state, i)
            if game.check_victory(silent=True): game_over = True; break
            engine.kalman_filter()
        game.detectives_moves.append(game.detectives_pos[:])
        if game_over: break
        bp, bt = mrx.search()
        game.x_automated_turn(bp, bt)
        if game.check_victory(silent=True): break
        engine.update_belief_after_mrx_move(bt)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
    return game.winner


def eval_winrates(model, n_games=50, mrx_explo=15, mrx_sims=25):
    """Paired comparison GNN argmax vs heuristic. Ritorna (gnn_winrate, heu_winrate, delta_pp)."""
    g_wins, h_wins = 0, 0
    for seed in range(n_games):
        random.seed(seed); np.random.seed(seed)
        if play_heuristic(mrx_explo, mrx_sims) == 0: h_wins += 1
        random.seed(seed); np.random.seed(seed)
        if play_one_argmax(model, mrx_explo, mrx_sims) == 0: g_wins += 1
    return g_wins / n_games, h_wins / n_games, (g_wins - h_wins) / n_games * 100

## 11. Training loop

In [ ]:
# Hyperparameters PPO
TOTAL_UPDATES = 100        # totale aggiornamenti PPO
GAMES_PER_UPDATE = 8       # episodi raccolti prima di ogni update
PPO_EPOCHS = 4
MINIBATCH_SIZE = 128
LR = 3e-4
WD = 1e-5
EVAL_EVERY = 10            # eval ogni N update
EVAL_N_GAMES = 50          # partite paired per eval intermedio (rapido)
MRX_EXPLO = 15
MRX_SIMS = 25

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
best_path = os.path.join(PPO_CKPT_DIR, f'rgnn_ppo_best_{RUN_TAG}.pt')
history = []
best_delta = -1e9

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
buf = RolloutBuffer()

# Eval iniziale (post BC, pre-PPO) — baseline contro cui misurare il progresso
print('=' * 60)
print('Eval pre-PPO (BC checkpoint, argmax, paired vs heuristic)')
print('=' * 60)
gnn_wr0, heu_wr0, delta0 = eval_winrates(model, n_games=EVAL_N_GAMES, mrx_explo=MRX_EXPLO, mrx_sims=MRX_SIMS)
print(f'GNN: {gnn_wr0*100:.1f}%  |  Heuristic: {heu_wr0*100:.1f}%  |  Delta: {delta0:+.1f}pp')
best_delta = delta0
history.append({'update': 0, 'gnn_wr': gnn_wr0, 'heu_wr': heu_wr0, 'delta_pp': delta0,
                'pi_loss': None, 'v_loss': None, 'entropy': None, 'approx_kl': None, 'clip_frac': None,
                'mean_episode_reward': None})
torch.save({
    'model_state_dict': model.state_dict(),
    'update': 0,
    'gnn_wr': float(gnn_wr0), 'heu_wr': float(heu_wr0), 'delta_pp': float(delta0),
    'config': cfg,
    'rtg_mean': BC_RTG_MEAN, 'rtg_std': BC_RTG_STD,
    'hyperparameters': {
        'GAMMA': GAMMA, 'GAE_LAMBDA': GAE_LAMBDA,
        'PPO_CLIP_EPS': PPO_CLIP_EPS, 'PPO_ENTROPY_COEF': PPO_ENTROPY_COEF,
        'PPO_VALUE_COEF': PPO_VALUE_COEF, 'LR': LR,
        'GAMES_PER_UPDATE': GAMES_PER_UPDATE, 'PPO_EPOCHS': PPO_EPOCHS,
        'MINIBATCH_SIZE': MINIBATCH_SIZE,
    },
}, best_path)
print(f'Initial BC baseline saved as current best: {best_path}')

t_start = time.time()
for upd in range(1, TOTAL_UPDATES + 1):
    # ----- Rollouts -----
    buf.clear()
    ep_winners = []
    ep_returns = []
    for _ in range(GAMES_PER_UPDATE):
        transitions, winner = collect_episode(model, mrx_explo=MRX_EXPLO, mrx_sims=MRX_SIMS)
        if transitions:
            buf.add_episode(transitions, last_value=0.0)
            ep_returns.append(sum(t['reward'] for t in transitions))
            ep_winners.append(winner)

    if len(buf) == 0:
        print(f'Update {upd}: 0 transizioni raccolte, skip')
        continue

    # ----- PPO update -----
    buf_tensors = buf.to_tensors(DEVICE)
    metrics = ppo_update(model, optimizer, buf_tensors, n_epochs=PPO_EPOCHS, minibatch_size=MINIBATCH_SIZE)

    rollout_wr = sum(1 for w in ep_winners if w == 0) / max(1, len(ep_winners))
    mean_ret = float(np.mean(ep_returns)) if ep_returns else 0.0
    line = (f'Upd {upd:3d}/{TOTAL_UPDATES}  '
            f'rollout_wr={rollout_wr*100:5.1f}%  ep_ret={mean_ret:+.2f}  '
            f'pi={metrics["pi_loss"]:+.3f} v={metrics["v_loss"]:.3f} '
            f'H={metrics["entropy"]:.3f} KL={metrics["approx_kl"]:+.4f} clip={metrics["clip_frac"]:.2f}')

    if upd % EVAL_EVERY == 0 or upd == TOTAL_UPDATES:
        gnn_wr, heu_wr, delta = eval_winrates(model, n_games=EVAL_N_GAMES, mrx_explo=MRX_EXPLO, mrx_sims=MRX_SIMS)
        line += f'  |  EVAL gnn={gnn_wr*100:.1f}% heu={heu_wr*100:.1f}% delta={delta:+.1f}pp'
        if delta > best_delta:
            best_delta = delta
            torch.save({
                'model_state_dict': model.state_dict(),
                'update': upd,
                'gnn_wr': float(gnn_wr), 'heu_wr': float(heu_wr), 'delta_pp': float(delta),
                'config': cfg,
                'rtg_mean': BC_RTG_MEAN, 'rtg_std': BC_RTG_STD,
                'hyperparameters': {
                    'GAMMA': GAMMA, 'GAE_LAMBDA': GAE_LAMBDA,
                    'PPO_CLIP_EPS': PPO_CLIP_EPS, 'PPO_ENTROPY_COEF': PPO_ENTROPY_COEF,
                    'PPO_VALUE_COEF': PPO_VALUE_COEF, 'LR': LR,
                    'GAMES_PER_UPDATE': GAMES_PER_UPDATE, 'PPO_EPOCHS': PPO_EPOCHS,
                    'MINIBATCH_SIZE': MINIBATCH_SIZE,
                },
            }, best_path)
            line += '  *NEW BEST*'
    else:
        gnn_wr = heu_wr = delta = None

    history.append({
        'update': upd, 'gnn_wr': gnn_wr, 'heu_wr': heu_wr, 'delta_pp': delta,
        'rollout_wr': rollout_wr, 'mean_episode_reward': mean_ret,
        **metrics,
    })
    print(line)

elapsed = time.time() - t_start
print(f'\nTotal training time: {elapsed/60:.1f} min')
print(f'Best delta: {best_delta:+.1f}pp (vs pre-PPO {delta0:+.1f}pp). Best checkpoint: {best_path}')

hist_df = pd.DataFrame(history)
hist_df.to_csv(os.path.join(PPO_CKPT_DIR, f'rgnn_ppo_history_{RUN_TAG}.csv'), index=False)

## 12. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 1. Winrate evolution
ax = axes[0, 0]
eval_df = hist_df.dropna(subset=['gnn_wr']).copy()
ax.plot(eval_df['update'], eval_df['gnn_wr']*100, 'o-', label='GNN argmax', linewidth=2)
ax.plot(eval_df['update'], eval_df['heu_wr']*100, 's--', label='Heuristic', alpha=0.7)
ax.axhline(eval_df['heu_wr'].iloc[0]*100, color='gray', linestyle=':', alpha=0.5, label='heu @ start')
ax.set_title('Detective winrate vs full Mr.X (paired)'); ax.set_xlabel('PPO update')
ax.set_ylabel('winrate (%)'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. Delta
ax = axes[0, 1]
ax.plot(eval_df['update'], eval_df['delta_pp'], 'o-', color='green', linewidth=2)
ax.axhline(0, color='black', alpha=0.3)
ax.set_title('Delta GNN - Heuristic (pp)'); ax.set_xlabel('PPO update'); ax.set_ylabel('pp')
ax.grid(True, alpha=0.3)

# 3. Rollout winrate (more frequent signal)
ax = axes[0, 2]
train_df = hist_df.dropna(subset=['rollout_wr']).copy()
ax.plot(train_df['update'], train_df['rollout_wr']*100, '-', alpha=0.6, label='rollout (sampling)')
if len(eval_df) > 0:
    ax.plot(eval_df['update'], eval_df['gnn_wr']*100, 'o-', label='eval (argmax)', linewidth=2)
ax.set_title('Rollout winrate vs eval winrate'); ax.set_xlabel('PPO update'); ax.set_ylabel('%')
ax.legend(); ax.grid(True, alpha=0.3)

# 4. Policy + Value loss
ax = axes[1, 0]
ax.plot(train_df['update'], train_df['pi_loss'], label='policy loss')
ax.set_title('Policy loss'); ax.set_xlabel('PPO update'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(train_df['update'], train_df['v_loss'], label='value MSE (normalized)', color='orange')
ax.set_title('Value loss'); ax.set_xlabel('PPO update'); ax.grid(True, alpha=0.3)

# 5. Entropy + KL
ax = axes[1, 2]
ax.plot(train_df['update'], train_df['entropy'], label='entropy', color='purple')
ax2 = ax.twinx()
ax2.plot(train_df['update'], train_df['approx_kl'], label='approx KL', color='red', alpha=0.7)
ax.set_title('Entropy (purple) + approx KL (red)'); ax.set_xlabel('PPO update')
ax.set_ylabel('entropy'); ax2.set_ylabel('KL')
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()